** Binary Connected Components Implementation ** 

In [53]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label
from collections import defaultdict
import json

F = np.loadtxt(
    "f_nearest_neighbors_lp.csv",
    delimiter=","
)

nRows = 42
nCols = 44

print(F.shape)

(1848, 20)


In [54]:
def get_matrix_u(F, tree):
    U = F.copy()

    for parent, children in tree.items():
        for child in children:
            U[:, parent] -= F[:, child]

    return U

In [55]:
## tree copy pasted from LP.ipynb 
trees = [ 
    {
	18: [19],
	19: [],
	7: [18, 17, 16],
	17: [],
	16: [],
	4: [7],
	3: [15, 14],
	15: [],
	14: [],
	2: [13],
	13: [],
	1: [9],
	9: [],
	0: [12, 11, 10, 8, 6, 5, 4, 3, 2, 1],
	12: [],
	11: [],
	10: [],
	8: [],
	6: [],
	5: []
}, 
    # original below
    {
	18: [19],
	19: [],
	7: [18, 17, 16],
	17: [],
	16: [],
	4: [7],
	3: [15, 14],
	15: [],
	14: [],
	2: [13],
	13: [],
	1: [9, 8],
	9: [],
	8: [],
	0: [12, 11, 10, 6, 5, 4, 3, 2, 1],
	12: [],
	11: [],
	10: [],
	6: [],
	5: []
}, ]

In [56]:
U = get_matrix_u(F, trees[0])

In [57]:

#generates binary connected components, 4 connectivity
def binary_connected_components(grid, threshold=0.1):
    binary = grid >= threshold

    # 4 connectivity means cells r connected if they share an edge (not including corners)
    structure = np.array([
        [0, 1, 0],
        [1, 1, 1],
        [0, 1, 0]
    ])

    labeled_grid, num_components = label(
        binary,
        structure=structure
    )

    return labeled_grid, num_components

In [58]:
def extract_boundary_edges(labeled_grid, component_id):
    rows, cols = labeled_grid.shape
    edges = []

    for r in range(rows):
        for c in range(cols):

            if labeled_grid[r, c] != component_id:
                continue

            # Top edge
            if r == 0 or labeled_grid[r-1, c] != component_id:
                edges.append(((c, r), (c+1, r)))

            # Bottom edge
            if r == rows-1 or labeled_grid[r+1, c] != component_id:
                edges.append(((c, r+1), (c+1, r+1)))

            # Left edge
            if c == 0 or labeled_grid[r, c-1] != component_id:
                edges.append(((c, r), (c, r+1)))

            # Right edge
            if c == cols-1 or labeled_grid[r, c+1] != component_id:
                edges.append(((c+1, r), (c+1, r+1)))

    return edges

In [59]:
### Boundary stitching 
def stitch_boundary_edges(edges):
    adjacency = defaultdict(list)

    # Map each vertex to its neighboring vertices
    for start, end in edges:
        adjacency[start].append(end)
        adjacency[end].append(start)

    unused = {
        frozenset((start, end))
        for start, end in edges
    }

    paths = []

    while unused:
        start, current = tuple(next(iter(unused)))
        path = [start, current]

        unused.remove(frozenset((start, current)))

        # Follow edges until we return to the start
        while current != start:
            for neighbor in adjacency[current]:
                edge = frozenset((current, neighbor))

                if edge in unused:
                    unused.remove(edge)
                    path.append(neighbor)
                    current = neighbor
                    break
            else:
                raise ValueError("Boundary path could not be closed")

        paths.append(path)

    return paths

In [60]:
boundaries = []

for component_id in range(1, num_components + 1):

    edges = extract_boundary_edges(
        labeled_grid,
        component_id
    )

    paths = stitch_boundary_edges(edges)

    boundaries.extend(paths)

boundaries

[[(21, 2),
  (22, 2),
  (22, 1),
  (21, 1),
  (21, 0),
  (20, 0),
  (19, 0),
  (18, 0),
  (18, 1),
  (19, 1),
  (19, 2),
  (20, 2),
  (21, 2)],
 [(23, 4),
  (24, 4),
  (24, 5),
  (25, 5),
  (25, 6),
  (25, 7),
  (24, 7),
  (24, 8),
  (24, 9),
  (25, 9),
  (25, 8),
  (26, 8),
  (27, 8),
  (28, 8),
  (28, 7),
  (29, 7),
  (30, 7),
  (30, 8),
  (30, 9),
  (30, 10),
  (31, 10),
  (31, 11),
  (32, 11),
  (32, 12),
  (33, 12),
  (34, 12),
  (34, 13),
  (35, 13),
  (35, 14),
  (36, 14),
  (36, 15),
  (36, 16),
  (37, 16),
  (37, 15),
  (37, 14),
  (38, 14),
  (38, 13),
  (39, 13),
  (40, 13),
  (40, 12),
  (40, 11),
  (40, 10),
  (39, 10),
  (39, 9),
  (39, 8),
  (38, 8),
  (38, 7),
  (38, 6),
  (37, 6),
  (37, 5),
  (36, 5),
  (35, 5),
  (34, 5),
  (34, 4),
  (33, 4),
  (33, 3),
  (32, 3),
  (32, 2),
  (31, 2),
  (30, 2),
  (30, 1),
  (29, 1),
  (28, 1),
  (27, 1),
  (26, 1),
  (26, 2),
  (25, 2),
  (24, 2),
  (23, 2),
  (23, 3),
  (22, 3),
  (22, 4),
  (23, 4)]]

In [62]:
threshold = 0.1
U = get_matrix_u(F, trees[0])

all_boundaries = {}

for clone_idx in range(U.shape[1]):

    grid = U[:, clone_idx].reshape(nRows, nCols)

    labeled_grid, num_components = (
        binary_connected_components(grid, threshold)
    )

    boundaries = []

    for component_id in range(1, num_components + 1):

        edges = extract_boundary_edges(
            labeled_grid,
            component_id
        )

        paths = stitch_boundary_edges(edges)

        boundaries.extend(paths)

    # Adjust coordinates for frontend grid padding
    all_boundaries[str(clone_idx)] = [
        [[x + 0.5, y + 0.5] for x, y in path]
        for path in boundaries
    ]

with open("cc_boundaries.json", "w") as file:
    json.dump(all_boundaries, file)